<a href="https://colab.research.google.com/github/AlicePF43/assistente-voz-ia/blob/main/assistente_voz_ia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
language = 'pt'

# 1. Gravação de Áudio Com Python (e Uma Pitada de JavaScript) 🎤

In [ ]:
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  # Executa o código JavaScript para gravar o áudio
  display(Javascript(RECORD))
  # Recebe o áudio gravado como resultado do JavaScript
  js_result = output.eval_js('record(%s)' % (sec * 1000))
   # Decodifica o áudio em base64
  audio = b64decode(js_result.split(',')[1])
  # Salva o áudio em um arquivo
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  # Retorna o caminho do arquivo de áudio (pasta padrão do Google Colab)
  return f'/content/{file_name}'

# Grava o áudio do usuário por um tempo determinado (padrão 5 segundos)
print('Ouvindo...\n')
record_file = record()

# Exibe o áudio gravado
display(Audio(record_file, autoplay=False))

# 2. Reconhecimento de Fala com Whisper (OpenAI) 🧠

In [ ]:
!pip install whisper transformers gTTS -q


In [ ]:
!pip install git+https://github.com/openai/whisper.git -q

In [ ]:
import whisper

# Selecione o modelo do Whisper que melhor atenda às suas necessidades:
# https://github.com/openai/whisper#available-models-and-languages
model = whisper.load_model("small")

# Transcreve o audio gravado anteriormente.
result = model.transcribe(record_file, fp16=False, language=language)
transcription = result["text"]
print(transcription)

ETAPA 3 — CRIAR FUNÇÃO DO ASSISTENTE

In [ ]:
import whisper
from transformers import pipeline
from gtts import gTTS
from IPython.display import Audio, display

language = 'pt'

# carrega modelos (uma vez só)
whisper_model = whisper.load_model("small")
generator = pipeline("text-generation", model="google/flan-t5-base")

def assistente():
    print("\n🎤 Fale alguma coisa...")

    record_file = record()

    # transcreve áudio
    result = whisper_model.transcribe(record_file, fp16=False, language=language)
    transcription = result["text"]

    print("📝 Você disse:", transcription)

    # comando de parada
    if "parar" in transcription.lower():
        print("🛑 Encerrando assistente...")
        return False

    # prompt estilo assistente
    prompt = f"""
    Responda como um assistente virtual de forma natural e curta:

    Usuário disse: {transcription}
    """

    resposta = generator(prompt, max_length=100)
    chatgpt_response = resposta[0]['generated_text']

    print("🤖 Assistente:", chatgpt_response)

    # gera áudio
    tts = gTTS(text=chatgpt_response, lang=language)
    tts.save("resposta.mp3")

    display(Audio("resposta.mp3", autoplay=True))

    return True

ETAPA 4 (LOOP CONTROLADO)

In [ ]:
for i in range(2):  # executa até 2 vezes
    continuar = assistente()
    if not continuar:
        break
